<a href="https://colab.research.google.com/github/minyi-k03/LargeLanguageModel/blob/Project-Based-Learning(PBL)/Gpt_Oss_20b_Reasoning_Test_GPQA_Diamond_using_ollama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# gpt-oss-20B 모델 추론 능력 테스트하기(GPQA Diamond 데이터셋)
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference :
1. https://openai.com/ko-KR/index/introducing-gpt-oss/
2. https://cookbook.openai.com/articles/gpt-oss/run-locally-ollama

## GPQA Diamond 데이터셋 : https://huggingface.co/datasets/Idavidrein/gpqa

In [1]:
!nvidia-smi

Wed Jan 28 06:12:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Ollama 설치


In [5]:
import subprocess
import time

#ztsd 설치
print("필수 도구(zstd) 설치 중...")
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Ollama 설치
print("\nOllama 설치 중...")
!curl -fsSL https://ollama.com/install.sh | sh

필수 도구(zstd) 설치 중...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.8 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,315 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease [24.3 kB]
Get:13 https://r

In [6]:
!ollama

Usage:
  ollama [flags]
  ollama [command]

Available Commands:
  serve       Start ollama
  create      Create a model
  show        Show information for a model
  run         Run a model
  stop        Stop a running model
  pull        Pull a model from a registry
  push        Push a model to a registry
  signin      Sign in to ollama.com
  signout     Sign out from ollama.com
  list        List models
  ps          List running models
  cp          Copy a model
  rm          Remove a model
  launch      Launch an integration with Ollama
  help        Help about any command

Flags:
  -h, --help      help for ollama
  -v, --version   Show version information

Use "ollama [command] --help" for more information about a command.


In [7]:
import subprocess

# Ollama 서버를 백그라운드에서 실행
subprocess.Popen("nohup ollama serve &", shell=True)

<Popen: returncode: None args: 'nohup ollama serve &'>

In [8]:
!ollama pull gpt-oss:20b

In [9]:
!ollama list

NAME           ID              SIZE     MODIFIED       
gpt-oss:20b    17052f91a42e    13 GB    22 seconds ago    


In [10]:
!curl http://localhost:11434

Ollama is running

In [11]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:11434/v1",  # Local Ollama API
    api_key="ollama"                       # Dummy key
)

response = client.chat.completions.create(
    model="gpt-oss:20b",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain what MXFP4 quantization is."}
    ]
)

print(response.choices[0].message.content)

**MXFP4 quantization – a quick‑look definition**

MXFP4 is a *format‑level* quantization technique that was introduced to give deep‑learning models a very low‑bit, highly‑compressible representation without losing too much accuracy.  
In practice it is:

| Symbol | What it stands for | Bit‑width | Purpose |
|--------|-------------------|-----------|---------|
| **MX** | “Mixed‑precision eXtended‑floating‑point” | — | The overarching scheme that lets the exponent and mantissa be treated independently. |
| **FP** | Floating‑point style representation | — | We keep the familiar *sign‑exponent‑mantissa* structure, but shrink all fields. |
| **4**  | Total payload bits per real number | **4** | The numeric value is encoded into only four bits, the smallest commonly used non‑trivial size that still gives a good trade‑off between range and precision. |

That means a single weight or activation is stored in **4 bits**. Internally the 4 bits are split into sub‑fields that mimic a “tiny” floatin

In [12]:
response = client.chat.completions.create(
    model="gpt-oss:20b",
    messages=[
        {"role": "system", "content": "당신은 도움이 되는 조수입니다."},
        {"role": "user", "content": "MXFP4 quantization이 무엇인지 설명해주세요."}
    ]
)

print(response.choices[0].message.content)

## MXFP4 Quantization이란?

### 1️⃣ 기본 개념: *Quantization*  
딥러닝 모델을 실제 제품에 배포할 때는  
- **전송 비용** (모델 크기),  
- **계산 비용** (Inference 속도),  
- **메모리 사용량**  
등을 줄이기 위해 가중치와 활성값을 **정밀도 낮은 형식**(bit‑width)으로 변환합니다.  
이 과정을 **Quantization**이라고 부릅니다.

| 정밀도 | 비트 수 | 대표 형식 | 용도 |
|--------|---------|-----------|------|
| Full‑Precision | 32 bit FP32 | IEEE‑754 | 학습 시 |
| Half‑Precision | 16 bit FP16 | IEEE‑754 | 학습 / inference |
| Integer | 8 bit INT8 | 스케일·제로포인트 | inference |
| 4‑bit | 4 bit | **FP4 / INT4 / MXFP4** | 메모리와 연산 부담 극대 감소 |
| 2‑bit | 2 bit | (심볼릭 등) | 실험적 활용 |

특히 4‑bit 레벨은 메모리 압축비가 **8배**에 이르며, 최근 GPU/TPU(특히 NVIDIA Ampere, AMD Instinct 등)의 **Tensor Core**에서도 4‑bit 연산을 가속화하려는 시도가 많아졌습니다.

---

### 2️⃣ MXFP4의 등장 배경

* **MXFP**(*Mixed‑Precision Floating‑Point*)는 4‑bit/8‑bit 등 **혼합 정밀도**를 지원하도록 설계한 고유 포맷입니다.  
* 4‑bit 부동소수점은 1개의 부호, 2개의 지수, 1개의 가정값을 갖습니다.  
    * 비트 구성 : `S|EEE|F` → `1|2|1` 비트  
* 기존의 4‑bit 정수형(INT4)과 달리 **부동소수점** 특성을 가지므로, **동적 범위**가 넓어져 대규모 모델(예: LLaMA/Meta‑LLama3, Mis

In [13]:
response = client.chat.completions.create(
    model="gpt-oss:20b",
    messages=[
        {"role": "system", "content": "당신은 도움이 되는 조수입니다."},
        {"role": "user", "content": "너는 한국어를 할 수 있니?"}
    ]
)

print(response.choices[0].message.content)

네, 저는 한국어로 소통할 수 있어요. 무엇이든 물어보세요!


In [15]:
gpqa_diamond_example_1 = """
두 개의 양자 상태 E1과 E2는 각각 10^-9초와 10^-8초의 수명을 가지고 있다.
이 두 에너지 준위를 명확하게 구분하고자 한다.
다음 보기 중 어떤 값이 이들이 명확히 분해될 수 있는 에너지 차이가 될 수 있는가?

A. 10^-4 eV
B. 10^-11 eV
C. 10^-8 eV
D. 10^-9 eV
"""

# 해설 :
# 불확정성 원리에 따르면, ΔE * Δt = ħ / 2 이다. 여기서 Δt는 수명이고, ΔE는 에너지 준위의 폭이다.
# Δt = 10^-9초일 때 → ΔE₁ = 3.3 × 10^-7 eV
# Δt = 10^-11초일 때 → ΔE₂ = 3.3 × 10^-8 eV 가 된다.

# 따라서 두 상태 사이의 에너지 차이는 10^-7 eV보다 훨씬 커야 한다.
# 따라서 정답은 10^-4 eV이다.

# 정답 : A. 10^-4 eV

In [16]:
response = client.chat.completions.create(
    model="gpt-oss:20b",
    messages=[
        {"role": "system", "content": "당신은 도움이 되는 조수입니다."},
        {"role": "user", "content": gpqa_diamond_example_1}
    ]
)

print(response.choices[0].message.content)

### Why 10⁻⁴ eV is the only viable choice

The ability to “clearly separate’’ two energy levels depends on how sharp each level is.  
A level’s natural linewidth is set by its lifetime through the energy–time
uncertainty principle

\[
\Delta E \,\Delta t \;\gtrsim\; \frac{\hbar}{2} .
\]

Here  
* \( \tau_{1}= 10^{-9}\,\text{s}\) (short lifetime → broad line)  
* \( \tau_{2}= 10^{-8}\,\text{s}\) (longer lifetime → narrower line)

#### 1. Compute the natural widths

Using  
\[
\hbar \;=\; 6.582 \times 10^{-16}\;\text{eV·s},
\]
the width of a level of lifetime \(\tau\) is  
\[
\Delta E = \frac{\hbar}{2\tau}.
\]

| Level | \( \tau \) | \(\Delta E\) (eV) |
|-------|------------|------------------|
| \(E_1\) | \(10^{-9}\) s | \(\displaystyle \frac{6.582\times10^{-16}}{2\times10^{-9}}\approx 3.3\times10^{-7}\) eV |
| \(E_2\) | \(10^{-8}\) s | \(\displaystyle \frac{6.582\times10^{-16}}{2\times10^{-8}}\approx 3.3\times10^{-8}\) eV |

The broader line (from the shorter lifetime) dominates the re

In [17]:
gpqa_diamond_example_2 = """
표준 모형을 다음과 같은 라그랑지언으로 확장한 이론을 고려하자:

L ⊂ i N̄ᴿ γ^μ ∂μ Nᴿ + (1/2)(∂^μ φ)^2 + |D^μ S|^2
  − (yᵢ / 2) φ N̄ᶜ{iR} Nᶜ_{iR} − g_{iα} N̄_{iR} L_α S − V(φ, S, H)

여기서 특이점 페르미온은 N_{iR} ∼ (1, 1, 0),
스칼라 더블릿은 S ∼ (1, 2, 1),
특이점 스칼라는 φ ∼ (1, 1, 0) 이다.

우리는 ⟨φ⟩² = x² + υ² 를 부여하며, ⟨φ⟩ = x, ⟨h⟩ = v 라고 둔다.

복사 보정을 통해 유사 골드스톤 보존 H₂의 질량에 대한 근사값은 얼마인가?

A. M_{h_{2}}^{2}=\frac{\left(x^{2}+v^{2}\right)}{8\pi^{2}}\left\{ \alpha_{1}M_{h_{1}}^{4}+\alpha_{2}M_{W}^{4}+\alpha_{3}M_{Z}^{4}-\alpha_{4}M_{t}^{4}+\alpha_{5}M_{H^{\pm}}^{4}+\alpha_{6}M_{H^{0}}^{4}+\alpha_{7}M_{A^{0}}^{4}-\alpha_{8}\sum M_{N_{i}}^{4}\right\}
B. M_{h_{2}}^{2}=\frac{1}{8\pi^{2}\left(x^{2}+v^{2}\right)}\left\{ \alpha_{1}M_{h_{1}}^{4}+\alpha_{2}M_{W}^{4}+\alpha_{3}M_{Z}^{4}-\alpha_{4}M_{t}^{4}+\alpha_{5}M_{H^{\pm}}^{4}+\alpha_{6}M_{H^{0}}^{4}-\alpha_{7}\sum M_{N_{i}}^{4}\right\}
C. M_{h_{2}}^{2}=\frac{1}{8\pi^{2}\left(x^{2}+v^{2}\right)}\left\{ \alpha_{1}M_{h_{1}}^{4}+\alpha_{2}M_{W}^{4}+\alpha_{3}M_{Z}^{4}+\alpha_{4}M_{H^{\pm}}^{4}+\alpha_{5}M_{H^{0}}^{4}+\alpha_{6}M_{A^{0}}^{4}-\alpha_{7}\sum M_{N_{i}}^{4}\right\}
D. M_{h_{2}}^{2}=\frac{1}{8\pi^{2}\left(x^{2}+v^{2}\right)}\left\{ \alpha_{1}M_{h_{1}}^{4}+\alpha_{2}M_{W}^{4}+\alpha_{3}M_{Z}^{4}-\alpha_{4}M_{t}^{4}+\alpha_{5}M_{H^{\pm}}^{4}+\alpha_{6}M_{H^{0}}^{4}+\alpha_{7}M_{A^{0}}^{4}-\alpha_{8}\sum M_{N_{i}}^{4}\right\}
"""

# 해설 :
# 유사 골드스톤 보존의 질량에 대한 근사는 벡터 보존 V, 무거운 힉스 보존 H, 그리고 페르미온에 대한 합으로 주어진다[1].
# 우리의 경우, 확장은 세 개의 특이점 게이지 페르미온, 하나의 이너트 더블릿, 그리고 하나의 스칼라 특이점으로 이루어져 있다.
# 여기에 우리는 추가로, 탑 쿼크 페르미온, Z 및 W 보존, 그리고 힉스 보존을 포함할 것이다.

# [1] https://journals.aps.org/prd/abstract/10.1103/PhysRevD.13.3333

# 정답 : D. M_{h_{2}}^{2}=\frac{1}{8\pi^{2}\left(x^{2}+v^{2}\right)}\left\{ \alpha_{1}M_{h_{1}}^{4}+\alpha_{2}M_{W}^{4}+\alpha_{3}M_{Z}^{4}-\alpha_{4}M_{t}^{4}+\alpha_{5}M_{H^{\pm}}^{4}+\alpha_{6}M_{H^{0}}^{4}+\alpha_{7}M_{A^{0}}^{4}-\alpha_{8}\sum M_{N_{i}}^{4}\right\}

<>:15: SyntaxWarning: invalid escape sequence '\l'
<>:15: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipython-input-3953058700.py:15: SyntaxWarning: invalid escape sequence '\l'
  A. M_{h_{2}}^{2}=\frac{\left(x^{2}+v^{2}\right)}{8\pi^{2}}\left\{ \alpha_{1}M_{h_{1}}^{4}+\alpha_{2}M_{W}^{4}+\alpha_{3}M_{Z}^{4}-\alpha_{4}M_{t}^{4}+\alpha_{5}M_{H^{\pm}}^{4}+\alpha_{6}M_{H^{0}}^{4}+\alpha_{7}M_{A^{0}}^{4}-\alpha_{8}\sum M_{N_{i}}^{4}\right\}


In [19]:
response = client.chat.completions.create(
    model="gpt-oss:20b",
    messages=[
        {"role": "system", "content": "당신은 도움이 되는 조수입니다."},
        {"role": "user", "content": gpqa_diamond_example_2}
    ]
)

print(response.choices[0].message.content)

**답: D**  

다음과 같이 코넬리–위르셰브 (Coleman‑Weinberg) 루프 교정이 주는 사전‑골드스톤 보존 \(H_{2}\) 의 질량의 근사식이 가장 일치합니다.  

\[
\boxed{M_{h_{2}}^{2}
   =\frac{1}{\,8\pi^{2}\,(x^{2}+v^{2})\,}
     \Bigl[
        \alpha_{1} M_{h_{1}}^{4}
      + \alpha_{2} M_{W}^{4}
      + \alpha_{3} M_{Z}^{4}
      - \alpha_{4} M_{t}^{4}
      + \alpha_{5} M_{H^{\pm}}^{4}
      + \alpha_{6} M_{H^{0}}^{4}
      + \alpha_{7} M_{A^{0}}^{4}
      - \alpha_{8}\sum_{i} M_{N_{i}}^{4}
     \Bigr] }   
\]

이 식은

- **양성** 기여 (스칼라 및 gauge boson 루프) 은 \(\,+\,\) 로,  
- **음성** 기여 (fermion 루프) 은 \(\,-\,\) 로 표시됩니다.  

이와 같이, 각 입자에 대한 부호와 계수 \(\alpha_{i}\) 가 모델에 맞춰 정해지는 점이 위 식의 핵심이며, 옵션 D 가 그에 부합합니다.


In [20]:
gpqa_diamond_example_3 = """
1,3-디브로모아다만탄(1,3-dibromoadamantane)을 과량의 KOH와 함께 240℃로 가열하면 연한 노란색 고체인 생성물 1이 만들어진다.
1의 ¹H NMR 스펙트럼은 다음과 같다:
4.79ppm(2H), 2.41~2.23ppm(10H), 1.94ppm(2H)
또한 IR 스펙트럼에서 1720cm⁻¹에 특징적인 흡수 밴드를 가진다.

이 생성물 1을 과량의 아이소프로폭사이드 알루미늄(aluminum isopropoxide)과 함께 가열하여 생성물 2를 만든다.

그 후, 오존을 -78℃에서 2의 용액에 기포로 통과시킨 후 디메틸설파이드(dimethylsulfide)를 첨가하여 생성물 3을 얻는다.

이때 생성물 3의 ¹H NMR 스펙트럼에서, 가장 쉴딩이 약한(가장 다운필드에 위치한) 수소 원자(중수소화 용매와 교환되는 수소 제외)의 커플링 패턴은 무엇인가?

A. triplet
B. triplet of triplets
C. pentet
D. doublet of triplets
"""

# 해설 :
# 1,3-디브로모아다만탄(1,3-dibromoadamantane)은 KOH와 반응하면 SN1 반응을 거쳐 3-브로모아다만탄-1-올(3-bromoadamantan-1-ol)을 형성한다.
# 이후 알코올의 수소가 또 다른 수산화이온(OH⁻)에 의해 탈양성자되면, Grob 분해(Grob fragmentation)가 일어나 생성물 1인 **7-메틸렌바이사이클로[3.3.1]노난-3-온(7-methylenebicyclo[3.3.1]nonan-3-one)**이 형성된다.

# 생성물 1을 아이소프로폭사이드 알루미늄(aluminum isopropoxide)과 반응시키면 케톤이 알코올로 환원되어 생성물 2인 **7-메틸렌바이사이클로[3.3.1]노난-3-올(7-methylenebicyclo[3.3.1]nonan-3-ol)**이 만들어진다.
# (구체적으로는 OH기가 케톤과 반대 방향을 향한 입체이성질체가 만들어지지만, 이는 이 문제의 해답에 영향을 주지 않는다.)
# 이 반응은 Meerwein–Ponndorf–Verley(MPV) 환원이라 불린다.

# 생성물 2에 오존을 -78℃에서 통과시키고 디메틸설파이드(dimethylsulfide)를 첨가하면, 알켄이 케톤으로 전환되어 생성물 3인 **7-하이드록시바이사이클로[3.3.1]노난-3-온(7-hydroxybicyclo[3.3.1]nonan-3-one)**이 형성된다.

# 가장 쉴딩이 약한(다운필드에 위치한) 교환되지 않는 수소 핵은 OH기가 결합된 같은 탄소(C1)에 결합된 수소(H1)이다.

# OH 수소는 중수소(D)와 쉽게 교환되기 때문에 H-H 커플링에는 관여하지 않는다.

# C1에는 두 개의 CH₂ 그룹이 결합되어 있고, 이들은 분자를 이등분하는 대칭 평면을 기준으로 대칭 파트너 관계에 있다.

# 바이사이클로논 골격은 고정된 체어 형태를 가지므로, H1과 CH₂ 수소들 사이의 3-결합 커플링은 축(axial) 방향과 평면(equatorial) 방향의 CH₂ 수소들에 대해 각각 다르게 나타난다. 각 방향에 수소가 2개씩 존재한다.

# 따라서 H1은 서로 다른 커플링 상수를 가진 2개의 2중 수소 집단과 커플링되므로, **triplet of triplets (삼중선의 삼중선)**의 형태로 분리된다.


# [1] https://journals.aps.org/prd/abstract/10.1103/PhysRevD.13.3333

# 정답 : B. triplet of triplets

In [21]:
response = client.chat.completions.create(
    model="gpt-oss:20b",
    messages=[
        {"role": "system", "content": "당신은 도움이 되는 조수입니다."},
        {"role": "user", "content": gpqa_diamond_example_3}
    ]
)

print(response.choices[0].message.content)

**Answer**

From the three‑dimensional fingerprint of the spectrum (IR, ¹H‑NMR and ¹³C‑NMR) the compound was identified as the **carboxylic‑acid product of the ozonolysis step**.  
Because the acid proton (‑COOH) is on a carbonyl carbon it does not couple to any other protons – the rapid exchange of the hydroxyl proton and the strong H‑bonding of a carboxylic acid prevent ordinary scalar coupling. Consequently the ¹H‑NMR signal of the acid proton is a **singlet** (typically appearing in the range 10–12 ppm).  

So, **the acid proton appears as a singlet in the ¹H‑NMR spectrum.**
